# Audio processing and voiceprint verification

**Owner:** Towi (Audio lead) · branch `feat/audio`

The voice model is the system's final gate. After the face model recognises someone, their voice has to (1) be confidently recognised and (2) belong to **the same member** before the product recommendation is shown.

| Section | What it covers | Rubric criterion |
|---|---|---|
| 1 | Inventory of every recording | Audio Quality |
| 2 | Loading and preprocessing | Audio Quality |
| 3 | Waveform + spectrogram per member | Audio Quality & Visualization |
| 4 | Augmentation | Audio Augmentation |
| 5 | Feature extraction to `audio_features.csv` | Audio Feature Extraction |
| 6 | Voice model and evaluation | Model Implementation, Evaluation |
| 7 | Confidence threshold with outsiders | Multimodal Logic |
| 8 | Save the model, test it as the app will use it | System Simulation |

The reusable code lives in `src/audio_pipeline.py`, so the command-line app uses exactly the same steps as this notebook.

In [ ]:
import sys, warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import soundfile as sf
import librosa, librosa.display
from sklearn.metrics import ConfusionMatrixDisplay

from src import audio_pipeline as ap

FIG_DIR = ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")
print("Audio folder:", ap.AUDIO_DIR)

## 1. Inventory: what did each phone record?

A recording is a long list of numbers: the air pressure at each instant. The **sample rate** is how many of those snapshots are taken per second. Think of a flipbook: 16 kHz means 16,000 pages a second, 48 kHz means 48,000. Different phones and apps (voice recorder, WhatsApp) use different rates and channel counts. Before comparing four people's voices, we check that we know exactly what we've got.

`soundfile.info` reads a file's header without loading the sound. A file that can't be read here is usually a compressed voice note that was renamed to `.wav` without being converted.

In [ ]:
rows = []
for path in sorted(ap.AUDIO_DIR.glob("*/*.wav")):
    try:
        info = sf.info(path)
        rows.append({"member": path.parent.name, "file": path.name, "sample_rate": info.samplerate,
                     "channels": info.channels, "duration_s": round(info.duration, 2), "readable": True})
    except Exception as err:
        rows.append({"member": path.parent.name, "file": path.name, "readable": False, "error": str(err)[:60]})

inventory = pd.DataFrame(rows)
display(inventory)
print(inventory.groupby("member").size().rename("clips per member").to_string())

**Interpretation:** _(fill in after running)_ How many clips does each member have? Do the sample rates and channel counts differ between members? If they do, that's the reason for section 2: resample everything to one standard format.

## 2. Loading and preprocessing

`ap.load_clip()` does three things to every clip, in this order:

1. **Resample to 16 kHz mono.** Every flipbook is redrawn at the same speed, as one channel. 16 kHz covers the full range of human speech (phone calls use 8 kHz), so nothing useful is lost, and the files stay small.
2. **Trim silence** (`librosa.effects.trim`, `top_db=20`). Anything more than 20 dB quieter than the loudest point at the start and end is cut. We tested 30, 25 and 20 dB on real clips: phone background noise sits only about 30 dB below speech, so 30 dB trimmed almost nothing. At 20 dB every clip was cut cleanly to where the speech starts (about 0.7 to 1 s in). The trade-off is that the very softest end of a word can be clipped, which carries little information about who is speaking. Without this, the second of silence before someone speaks would be averaged into their "voice" features, and people who paused longer would look different for the wrong reason.
3. **Peak-normalise the volume** (loudest point = 0.9). How loud a clip is depends mostly on how close the phone was, not on who's speaking. If we left it, the model could learn "loud = Kethia", which breaks the moment she holds her phone further away. The 0.9 (not 1.0) leaves headroom so adding noise in section 4 doesn't distort the sound.

The plot below shows one clip before and after preprocessing.

In [ ]:
example = sorted(ap.AUDIO_DIR.glob("*/*yes_approve.wav"))[0]
raw, sr_raw = librosa.load(example, sr=None, mono=True)
clean = ap.load_clip(example)

fig, axes = plt.subplots(1, 2, figsize=(13, 3), sharey=True)
librosa.display.waveshow(raw, sr=sr_raw, ax=axes[0])
axes[0].set(title=f"Raw: {example.name} ({sr_raw} Hz, {len(raw)/sr_raw:.2f} s)", xlabel="Time (s)", ylabel="Amplitude")
librosa.display.waveshow(clean, sr=ap.SR, ax=axes[1], color="tab:orange")
axes[1].set(title=f"Preprocessed: 16 kHz, trimmed, normalised ({len(clean)/ap.SR:.2f} s)", xlabel="Time (s)")
plt.tight_layout(); plt.savefig(FIG_DIR / "audio_preprocessing.png", dpi=150); plt.show()

**Interpretation:** _(fill in after running)_ How much silence was trimmed? How did the height of the wave change after normalising?

## 3. Waveform and spectrogram for each member

The two views show different things:

- **Waveform:** loudness over time. You can see where each word starts and stops, but not *what the sound is made of*.
- **Mel spectrogram:** which pitches are present at each moment, like a piano roll. Time runs left to right, pitch goes up, and brightness is how strong that pitch is. The mel scale spaces the pitches the way human ears hear them: lots of detail in the low range where voices live, less in the high range.

Horizontal stripes in voiced sounds are **harmonics**: the base pitch of the voice and its multiples. Their spacing is set by how fast the vocal cords vibrate, which is one of the things that makes each voice different.

In [ ]:
members = sorted(p.name for p in ap.AUDIO_DIR.iterdir() if p.is_dir())
fig, axes = plt.subplots(len(members), 2, figsize=(14, 3 * len(members)))
for row, member in zip(np.atleast_2d(axes), members):
    path = sorted((ap.AUDIO_DIR / member).glob("*yes_approve.wav"))[0]
    y = ap.load_clip(path)
    librosa.display.waveshow(y, sr=ap.SR, ax=row[0])
    row[0].set(title=f"{member}: waveform ('Yes, approve')", xlabel="Time (s)", ylabel="Amplitude")
    mel_db = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=ap.SR, n_mels=64), ref=np.max)
    img = librosa.display.specshow(mel_db, sr=ap.SR, x_axis="time", y_axis="mel", ax=row[1])
    row[1].set(title=f"{member}: mel spectrogram", xlabel="Time (s)", ylabel="Frequency (Hz, mel scale)")
    fig.colorbar(img, ax=row[1], format="%+2.0f dB")
plt.tight_layout(); plt.savefig(FIG_DIR / "audio_waveforms_spectrograms.png", dpi=150); plt.show()

**Interpretation:** _(fill in after running, 3 to 4 sentences)_
- Can you see the two words "yes" and "approve" as separate bursts in each waveform?
- In the spectrograms, where is most of the energy (low or high frequencies)? Whose harmonic stripes are spaced further apart (a higher voice)?
- Which differences between members look like *voice* differences, and which look like *recording* differences (background noise as a haze across the whole spectrogram, for example)?

## 4. Augmentation

With 4 recordings per person, the model sees very little variety. Augmentation creates realistic variations of each recording, so the model learns what stays constant about a voice instead of memorising four specific clips.

| Augmentation | What it simulates | Why it's kept small |
|---|---|---|
| Pitch +1 / -1 semitone | a slightly higher or lower voice (tired, excited) | A large shift makes you sound like a different person, and pitch is part of what identifies you. One semitone is about the natural day-to-day variation. |
| Speed x1.1 / x0.9 | talking a bit faster or slower, with pitch unchanged | Speed changes the timing but not the voice itself. |
| Background noise at 20 dB SNR | a noisier room | The speech stays 100 times more powerful than the noise, which sounds like a busy room, not a broken recording. |

So each recording becomes 6 versions (original + 5). That's **5 augmentations per sample**; the rubric asks for at least 2.

One important rule follows from this: an augmented copy is almost the same recording. If the original is in training and its copy in testing, the test is too easy. Section 6 keeps every recording and all its copies on the same side of the split.

In [ ]:
y = ap.load_clip(example)
versions = ap.augment(y, seed=0)
fig, axes = plt.subplots(2, 3, figsize=(15, 6.5), sharey=True)
for ax, (name, y_aug) in zip(axes.flat, versions.items()):
    mel_db = librosa.power_to_db(librosa.feature.melspectrogram(y=y_aug, sr=ap.SR, n_mels=64), ref=np.max)
    librosa.display.specshow(mel_db, sr=ap.SR, x_axis="time", y_axis="mel", ax=ax)
    ax.set(title=f"{name} ({len(y_aug)/ap.SR:.2f} s)", xlabel="Time (s)", ylabel="Hz (mel)")
plt.suptitle(f"Augmentations of {example.name}", y=1.0)
plt.tight_layout(); plt.savefig(FIG_DIR / "audio_augmentations.png", dpi=150); plt.show()

**Interpretation:** _(fill in after running)_ Compare each version with the original. Pitch shifts move the harmonic stripes slightly up or down. Speed changes stretch or squeeze the clip in time. Noise adds an even, speckled texture across all frequencies and drowns out the quietest details of the speech. The overall pattern of the words should still be recognisable in every version, which is the point.